# Training QET with disk-backed datasets

This example exercises MatGL's large-dataset path end to end: `write_mgl_shards`, `MGLDiskDataset`, `ShardBatchSampler`, and `MGLDataModule` feed a QET potential trained with PyTorch Lightning.

> **Important:** the small LiF dataset below is synthetic and exists only to demonstrate the API. Replace `iter_records` with a streaming reader for real DFT structures, energies, forces, stresses, and charges before using the model scientifically.

## Imports and reproducibility

QET training uses force derivatives, so Lightning must run with `inference_mode=False`. The example disables optional Warp kernels to remain portable on CPU-only installations.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import lightning as L
import torch
from pymatgen.core import Lattice, Structure

from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.data import MGLDataModule, MGLDiskDataset, ShardBatchSampler, write_mgl_shards
from matgl.models import QET
from matgl.utils.maths import scatter_add
from matgl.utils.training import PotentialLightningModule

torch.manual_seed(7)
torch.set_float32_matmul_precision("high")

# TemporaryDirectory keeps the example repeatable and removes its shards when the kernel exits.
workspace = tempfile.TemporaryDirectory(prefix="matgl_qet_disk_")
dataset_root = Path(workspace.name)
dataset_root

## Stream records into transactional shards

A record contains a pymatgen structure and a consistent `labels` mapping. Node labels (`forces`, `charges`) may have different leading dimensions across structures; graph labels (`energies`, `stresses`) must keep a consistent shape. `include_ref_charge=True` also stores charges as `graph.q_ref`, whose per-graph sum constrains QEq.

In [ ]:
element_types = ("Li", "F")
converter = Structure2Graph(element_types=element_types, cutoff=5.0)


def iter_records(count: int):
    """Yield toy neutral LiF configurations without retaining the dataset in memory."""
    for index in range(count):
        displacement = 0.005 * (index - count / 2)
        structure = Structure(
            Lattice.cubic(4.0 + 0.01 * index),
            element_types,
            [[0.0, 0.0, 0.0], [0.5 + displacement, 0.5, 0.5]],
        )
        forces = torch.zeros((2, 3))
        forces[1, 0] = -displacement
        yield {
            "structure": structure,
            "labels": {
                "energies": float(displacement**2),
                "forces": forces,
                "stresses": torch.zeros(6),
                "charges": torch.tensor([0.5, -0.5]),
            },
        }


for split, count in {"train": 16, "valid": 4, "test": 4}.items():
    write_mgl_shards(
        iter_records(count),
        dataset_root / split,
        converter=converter,
        shard_size=4,
        include_ref_charge=True,
    )

train_dataset = MGLDiskDataset(dataset_root / "train")
print(f"{len(train_dataset)=}, {train_dataset.num_shards=}")
print(train_dataset.label_schema)

## Inspect shard-local batching

The sampler randomizes shards and records, but one batch never crosses a shard boundary. In distributed training it assigns disjoint shards to ranks and balances their optimizer-step counts.

In [ ]:
sampler = ShardBatchSampler(train_dataset, batch_size=4, shuffle=True, seed=7)
first_epoch = list(sampler)
print("First three batches:", first_epoch[:3])
assert all(len({train_dataset._location(index)[0] for index in batch}) == 1 for batch in first_epoch)

## Configure the Lightning DataModule and QET

`MGLDataModule` opens only the splits required by the current Lightning stage and automatically selects MatGL's PES collator from the manifest label keys. For production, increase `num_workers`, choose a shard size that amortizes deserialization without exhausting worker memory, and create at least as many shards as distributed ranks.

In [ ]:
data_module = MGLDataModule(
    dataset_root,
    batch_size=4,
    num_workers=0,  # use multiple workers for a real large dataset
    seed=7,
)
data_module.setup("fit")
assert isinstance(data_module.train_dataloader().batch_sampler, ShardBatchSampler)

model = QET(
    element_types=element_types,
    units=16,
    nblocks=1,
    num_rbf=16,
    cutoff=5.0,
    use_warp=False,
)
module = PotentialLightningModule(
    model=model,
    energy_weight=1.0,
    force_weight=1.0,
    stress_weight=0.0,
    charge_weight=0.1,
    lr=1e-3,
    loss="mse_loss",
)

## Train and test

This demonstration uses two short epochs. Real training requires physically meaningful labels, convergence monitoring, held-out validation, and appropriate element-energy references.

In [ ]:
trainer = L.Trainer(
    max_epochs=2,
    accelerator="cpu",
    inference_mode=False,
    logger=False,
    enable_checkpointing=False,
    default_root_dir=dataset_root / "lightning",
)
trainer.fit(module, datamodule=data_module)
trainer.test(module, datamodule=data_module)

## Verify the QEq charge constraint

Because every stored graph has `q_ref`, QET uses its per-graph sum as the total-charge constraint. The individual predicted charges are learned, while their sum matches the reference total.

In [ ]:
data_module.setup("test")
batch = next(iter(data_module.test_dataloader()))
graph = batch[0]
with torch.enable_grad():
    module.step(batch)

predicted_charge = module._last_preds[3].detach()
predicted_total = scatter_add(predicted_charge, graph.batch, dim_size=graph.num_graphs)
reference_total = scatter_add(graph.q_ref, graph.batch, dim_size=graph.num_graphs)
print("Predicted totals:", predicted_total)
print("Reference totals:", reference_total)
assert torch.allclose(predicted_total, reference_total, atol=1e-6)

## Scaling to a real dataset

- Make `iter_records` stream from JSONL, a database cursor, or another lazy source; do not first construct a list of every structure.
- Keep label keys consistent within each split. Use eV, eV/Å, and MatGL's documented stress convention.
- Set `shard_size` so one loaded shard comfortably fits in each worker process.
- Use `num_workers > 0`, pinned memory on CUDA, and at least one shard per distributed rank.
- Precomputed shards use pickle-backed PyTorch loading and must come from a trusted source.
- Validate energies, forces, stresses, charges, and transferability against independent reference data before deploying the trained potential.